In [5]:
import numpy as np
from astropy.table import Table
import h5py

In [10]:
# Relevant parameters from Li+(25) XP modelling
msms = h5py.File("../../BigData/msms_36m_v0.hdf5", "r")
msms["table"]["columns"].keys()
new_table = Table()
labels = ["source_id", "mass_single", "mh_single"]
for label in labels:
    new_table.add_column(np.array(msms["table"]["columns"][label]["data"]), name=label)
    
new_df = new_table.to_pandas()

gaia_hdf5 = Table.read("../../BigData/ALL_gaia_source_and_xp_within_200pc.hdf5", format="hdf5")
new_gaia_table = Table()
labels = ["source_id", "ra", "dec", "parallax", "pmra", "pmdec", "phot_g_mean_mag", "bp_rp"]
for label in labels:
    new_gaia_table.add_column(np.array(gaia_hdf5[label]), name=label)

new_gaia_df = new_gaia_table.to_pandas()

merged_df = new_gaia_df.merge(new_df, on="source_id", how="inner")
parent_table = Table.from_pandas(merged_df)

parent_table["mass"] = parent_table["mass_single"]
parent_table.remove_column("mass_single")

In [11]:
parent_table = parent_table[(parent_table["mass"] > 0.8) & (parent_table["mass"] < 1.2)] # solar range
print(len(parent_table))

177824


In [12]:
parent_table.write("./solar_mass_primaries.fits", format="fits", overwrite=True)